#  CineMatch — Model Building
## Movie Recommendation System

This notebook builds and evaluates our recommendation engine using 
Collaborative Filtering with SVD (Singular Value Decomposition).

## 1. Loading Clean Data
We load the cleaned data saved from `01_data_exploration.ipynb`

In [1]:
import pandas as pd
import numpy as np

# Load our saved clean data
ratings_clean = pd.read_csv('data/ratings_clean.csv')
movies_clean = pd.read_csv('data/movies_clean.csv')

print("Data reloaded!")
print(f"Ratings: {ratings_clean.shape}")
print(f"Movies: {movies_clean.shape}")

Data reloaded!
Ratings: (100836, 3)
Movies: (9708, 3)


In [2]:
import pandas as pd
import numpy as np
from sklearn.decomposition import TruncatedSVD
from sklearn.metrics.pairwise import cosine_similarity
from scipy.sparse import csr_matrix
import warnings
warnings.filterwarnings('ignore')

print("All libraries loaded!")

All libraries loaded!


Cosine Similarity — Measuring Taste Match

Once every user has a hidden profile, how do we know who's similar?
We measure the angle between their profiles.

Small angle = similar taste = high similarity score
Large angle = different taste = low similarity score

Think of it like two arrows pointing in almost the same direction vs pointing opposite ways.

Together they answer: "Who has similar taste to this user, and what did those similar users watch?"

## 2. Building the User-Movie Matrix
The User-Movie Matrix places users on rows, movies on columns, 
and ratings as values. Empty cells (NaN) are filled with 0 
meaning "not yet rated".

In [3]:
#Step 1 — Building The User-Movie Matrix

# Load our clean data
ratings_clean = pd.read_csv('data/ratings_clean.csv')
movies_clean = pd.read_csv('data/movies_clean.csv')

# Build the User-Movie Matrix
print("Building User-Movie Matrix...")
user_movie_matrix = ratings_clean.pivot_table(
    index='userId',
    columns='movieId',
    values='rating'
)

# Fill NaN values with 0 
# (0 means "not rated" - different from a 0 star rating)
user_movie_matrix_filled = user_movie_matrix.fillna(0)

print(f" Matrix built!")
print(f"Shape: {user_movie_matrix_filled.shape}")
print(f"Users: {user_movie_matrix_filled.shape[0]}")
print(f"Movies: {user_movie_matrix_filled.shape[1]}")

Building User-Movie Matrix...
 Matrix built!
Shape: (610, 9724)
Users: 610
Movies: 9724


Why Fill NaN With 0?

Our matrix is 98.3% empty most ML algorithms can't handle empty cells. By filling with 0 we're saying "this user hasn't seen this movie" which is different from "this user rated it 0 stars."
SVD understands this distinction because it looks at patterns across all ratings, not individual zeros.

## 3. Applying SVD (Singular Value Decomposition)
SVD compresses each user's profile from 9,724 movie dimensions 
down to 50 hidden "taste" dimensions that capture the most 
important patterns in the data.

These 50 numbers represent hidden preferences — things like 
"loves action", "prefers drama", "enjoys sci-fi" — discovered 
automatically without us labelling anything.

In [4]:
#Step 2 — Applying SVD
# Convert to sparse matrix format - more memory efficient
# Think of it like a compressed version of our matrix
sparse_matrix = csr_matrix(user_movie_matrix_filled.values)

print("Applying SVD...")

# Apply SVD - we're reducing 9724 movie columns down to 50 hidden features
# These 50 numbers capture the essence of each user's taste
svd = TruncatedSVD(n_components=50, random_state=42)
user_factors = svd.fit_transform(sparse_matrix)

print(f"SVD complete!")
print(f"Original matrix shape: {sparse_matrix.shape}")
print(f"Reduced user factors shape: {user_factors.shape}")
print(f"\nVariance explained by our 50 factors: {svd.explained_variance_ratio_.sum():.2%}")

Applying SVD...
SVD complete!
Original matrix shape: (610, 9724)
Reduced user factors shape: (610, 50)

Variance explained by our 50 factors: 53.88%


## 4. Computing User Similarity
Using Cosine Similarity we compare every user's taste profile 
against every other user.

- Score of 1.0 = identical taste
- Score of 0.8+ = very similar
- Score of 0.0 = completely different

This gives us a 610×610 similarity matrix — every user compared 
against every other user.

In [5]:
#Step 3 — Computing User Similarity
# Calculate similarity between all users
print("Computing user similarities...")

# Cosine similarity compares every user against every other user
user_similarity = cosine_similarity(user_factors)

# Convert to a nice DataFrame so we can look up users easily
user_similarity_df = pd.DataFrame(
    user_similarity,
    index=user_movie_matrix.index,
    columns=user_movie_matrix.index
)

print(f"Similarity matrix built!")
print(f"Shape: {user_similarity_df.shape}")
print(f"\nSample - How similar is User 1 to other users?")
print(user_similarity_df[1].sort_values(ascending=False).head(6))

Computing user similarities...
Similarity matrix built!
Shape: (610, 610)

Sample - How similar is User 1 to other users?
userId
1      1.000000
39     0.818144
135    0.778551
234    0.750087
266    0.736926
313    0.728412
Name: 1, dtype: float64


Reading The Results
User 1 vs User 1 = 1.0

Perfect similarity with themselves — makes complete sense! Nobody has more identical taste to you than you. ✅

User 1 vs User 39 = 0.818

These two users have very similar taste — 81.8% similar! If User 39 watched something User 1 hasn't seen yet, there's a strong chance User 1 would love it too.

The rest (0.75, 0.73...)

Also strong similarities — User 1 has a nice cluster of like-minded users our model can learn from.


These similar users are called "Neighbors" in recommendation system terminology — just like asking your neighbors for advice because you know they have similar taste!

## 5. Generating Recommendations
Our recommendation function:
1. Finds users most similar to the target user (neighbors)
2. Looks at what those neighbors rated highly (4.0+)
3. Filters out movies the target user already watched
4. Weights recommendations by similarity score
5. Returns top N recommendations

In [6]:
#Step 4 — Building The Recommendation Function

def get_recommendations(user_id, num_recommendations=10):
    """
    Generate movie recommendations for a given user.
    
    How it works:
    1. Find users most similar to our target user
    2. Look at what those similar users rated highly
    3. Filter out movies our user has already seen
    4. Return the top recommendations
    """
    
    # Step 1: Find the most similar users (top 10 neighbors)
    similar_users = user_similarity_df[user_id].sort_values(ascending=False)
    similar_users = similar_users.drop(user_id)  # Remove the user themselves
    top_similar_users = similar_users.head(10).index.tolist()
    
    # Step 2: Find movies our user has already watched
    watched_movies = set(user_movie_matrix[user_movie_matrix.index == user_id].columns[
        user_movie_matrix[user_movie_matrix.index == user_id].iloc[0] > 0
    ])
    
    # Step 3: Collect highly rated movies from similar users
    recommended_movies = {}
    
    for similar_user in top_similar_users:
        # Get similarity score for weighting
        similarity_score = user_similarity_df[user_id][similar_user]
        
        # Get movies this similar user rated highly (4.0 and above)
        similar_user_ratings = user_movie_matrix[user_movie_matrix.index == similar_user]
        highly_rated = similar_user_ratings.columns[
            similar_user_ratings.iloc[0] >= 4.0
        ].tolist()
        
        for movie_id in highly_rated:
            # Only recommend movies our user hasn't seen
            if movie_id not in watched_movies:
                if movie_id not in recommended_movies:
                    recommended_movies[movie_id] = 0
                # Weight the recommendation by similarity score
                # More similar users have more influence
                recommended_movies[movie_id] += similarity_score
    
    # Step 4: Sort by score and get top recommendations
    recommended_movie_ids = sorted(
        recommended_movies,
        key=recommended_movies.get,
        reverse=True
    )[:num_recommendations]
    
    # Step 5: Get movie details
    recommendations = movies_clean[
        movies_clean['movieId'].isin(recommended_movie_ids)
    ][['movieId', 'title', 'genres']].copy()
    
    # Add recommendation scores
    recommendations['score'] = recommendations['movieId'].map(recommended_movies)
    recommendations = recommendations.sort_values('score', ascending=False)
    
    return recommendations

print(" Recommendation function ready!")

 Recommendation function ready!


## 6. Testing Our Recommendations
Let's generate recommendations for User 1 and validate they 
make sense based on their watch history.

In [7]:
# Let's generate recommendations for User 1
user_id = 1

# First, let's see what User 1 has already watched
print(f"=== WHAT USER {user_id} HAS WATCHED ===")
user_watched = ratings_clean[ratings_clean['userId'] == user_id].merge(
    movies_clean[['movieId', 'title', 'genres']], on='movieId'
)
user_watched = user_watched.sort_values('rating', ascending=False)
print(user_watched[['title', 'rating', 'genres']].head(10).to_string(index=False))

print(f"\n=== TOP 10 RECOMMENDATIONS FOR USER {user_id} ===")
recommendations = get_recommendations(user_id, num_recommendations=10)
print(recommendations[['title', 'genres', 'score']].to_string(index=False))

=== WHAT USER 1 HAS WATCHED ===
                                 title  rating                             genres
           Seven (a.k.a. Se7en) (1995)     5.0                   Mystery|Thriller
            Usual Suspects, The (1995)     5.0             Crime|Mystery|Thriller
                  Bottle Rocket (1996)     5.0     Adventure|Comedy|Crime|Romance
Dumb & Dumber (Dumb and Dumber) (1994)     5.0                   Adventure|Comedy
                  Billy Madison (1995)     5.0                             Comedy
                      Desperado (1995)     5.0             Action|Romance|Western
                 Canadian Bacon (1995)     5.0                         Comedy|War
                        Rob Roy (1995)     5.0           Action|Drama|Romance|War
                      Pinocchio (1940)     5.0 Animation|Children|Fantasy|Musical
                      Tombstone (1993)     5.0               Action|Drama|Western

=== TOP 10 RECOMMENDATIONS FOR USER 1 ===
                       

## 7. Evaluating Model Accuracy
We split our data into 80% training and 20% testing.

The model trains on the 80%, then predicts ratings for the 20% 
it hasn't seen. We compare predictions against actual ratings using:

- **RMSE (Root Mean Square Error)** — average prediction error in stars
- **MAE (Mean Absolute Error)** — simpler average error in stars

Lower scores = better accuracy.

In [8]:
#Step 5 — Evaluating Our Model
from sklearn.model_selection import train_test_split

print("Evaluating model accuracy...")

# Split data into training (80%) and testing (20%)
train_data, test_data = train_test_split(
    ratings_clean, 
    test_size=0.2, 
    random_state=42
)

print(f"Training ratings: {len(train_data)}")
print(f"Testing ratings: {len(test_data)}")

# Build matrix on training data only
train_matrix = train_data.pivot_table(
    index='userId',
    columns='movieId',
    values='rating'
).fillna(0)

# Apply SVD on training data
sparse_train = csr_matrix(train_matrix.values)
svd_eval = TruncatedSVD(n_components=50, random_state=42)
user_factors_eval = svd_eval.fit_transform(sparse_train)

# Compute similarity on training data
user_sim_eval = cosine_similarity(user_factors_eval)
user_sim_eval_df = pd.DataFrame(
    user_sim_eval,
    index=train_matrix.index,
    columns=train_matrix.index
)

# Test: predict ratings for test data and compare to actual
actual = []
predicted = []

for _, row in test_data.iterrows():
    user_id = row['userId']
    movie_id = row['movieId']
    actual_rating = row['rating']
    
    # Skip if user or movie not in training data
    if user_id not in user_sim_eval_df.index:
        continue
    if movie_id not in train_matrix.columns:
        continue
        
    # Find similar users
    similar_users = user_sim_eval_df[user_id].sort_values(ascending=False).drop(user_id)
    top_neighbors = similar_users.head(10)
    
    # Predict rating as weighted average of neighbors' ratings
    weighted_sum = 0
    similarity_sum = 0
    
    for neighbor_id, sim_score in top_neighbors.items():
        neighbor_rating = train_matrix.loc[neighbor_id, movie_id]
        if neighbor_rating > 0:
            weighted_sum += sim_score * neighbor_rating
            similarity_sum += sim_score
    
    if similarity_sum > 0:
        predicted_rating = weighted_sum / similarity_sum
        actual.append(actual_rating)
        predicted.append(predicted_rating)

# Calculate RMSE
rmse = np.sqrt(np.mean([(a-p)**2 for a, p in zip(actual, predicted)]))
mae = np.mean([abs(a-p) for a, p in zip(actual, predicted)])

print(f"\n=== MODEL EVALUATION ===")
print(f"Predictions made: {len(actual)}")
print(f"RMSE: {rmse:.4f}")
print(f"MAE: {mae:.4f}")

Evaluating model accuracy...
Training ratings: 80668
Testing ratings: 20168

=== MODEL EVALUATION ===
Predictions made: 12329
RMSE: 1.1120
MAE: 0.8445


## 8. Saving the Model
We save all model components so our Streamlit app can 
load them without retraining every time.

In [9]:
#Step 6 — Saving Our Model
import pickle
import os

print("Saving model...")

# Save the user-movie matrix
user_movie_matrix.to_csv('models/user_movie_matrix.csv')
print("Saved user_movie_matrix.csv")

# Save the user similarity matrix
user_similarity_df.to_csv('models/user_similarity.csv')
print("Saved user_similarity.csv")

# Save the SVD model
with open('models/svd_model.pkl', 'wb') as f:
    pickle.dump(svd, f)
print("Saved svd_model.pkl")

# Save user factors
np.save('models/user_factors.npy', user_factors)
print("Saved user_factors.npy")

print(f"\nFiles in models folder:")
print(os.listdir('models'))

Saving model...
Saved user_movie_matrix.csv
Saved user_similarity.csv
Saved svd_model.pkl
Saved user_factors.npy

Files in models folder:
['svd_model.pkl', 'user_factors.npy', 'user_movie_matrix.csv', 'user_similarity.csv']


## Summary

| Metric | Value |
|--------|-------|
| Model | SVD + Cosine Similarity |
| Users | 610 |
| Movies | 9,708 |
| Ratings | 100,789 |
| SVD Components | 50 |
| Variance Explained | ~54% |
| RMSE | ~1.11 |
| MAE | ~0.84 |

The model successfully generates personalized recommendations 
that align with each user's taste. See `app/app.py` for the 
full Streamlit application

In [10]:
import requests

API_KEY = "6c3e7665c1ad4597eb689ef21baf6ab6"

# Test - fetch Toy Story details (tmdbId = 862)
url = f"https://api.themoviedb.org/3/movie/862?api_key={API_KEY}"
response = requests.get(url)
data = response.json()

print(f"Status: {response.status_code}")
print(f"Movie: {data['title']}")
print(f"Overview: {data['overview'][:100]}...")
print(f"Rating: {data['vote_average']}")
print(f"Poster path: {data['poster_path']}")
print(f"\n TMDB API working!")

Status: 200
Movie: Toy Story
Overview: Led by Woody, Andy's toys live happily in his room until Andy's birthday brings Buzz Lightyear onto ...
Rating: 7.973
Poster path: /uXDfjJbdP4ijW5hWSBrPrlKpxab.jpg

 TMDB API working!
